# 03b — GMPE vs ShakeMap Spatial Comparison

Compares the four NGA-West2 GMPEs (ASK14, BSSA14, CB14, CY14) against the USGS ShakeMap (interpolated by nearest neighbor) for the 1994 Northridge earthquake.

Inputs:
- `output/gmpe_nga_west2/grid_gmpe_pga_sa1s.xlsx` — pre-computed Sa(1.0 s) grid with one sheet per GMPE (produced by `scripts/run_gmpe_nga_west2.py`).
- `data/grid.xml` — USGS ShakeMap v4 grid for the Northridge event.

Outputs (saved to `output/gmpe_nga_west2/`):
- `comparison_5panel_Sa1s.png` — five-panel spatial map: ASK14 · BSSA14 · CB14 · CY14 · ShakeMap.
- `comparison_residuals.png` — per-GMPE residual scatter (predicted − ShakeMap).
- `comparison_summary_stats.csv` — bias / MAE / RMSE / R² per GMPE.

Used in §3.2 (Kubilay's individual project) discussion of GMPE selection.

## 0. Preflight — auto-install missing dependencies

Run the cell below first. If it installs anything, restart the kernel before continuing.

In [ ]:
import sys, importlib, subprocess
_REQUIRED = ['numpy', 'pandas', 'matplotlib', 'scipy', 'openpyxl']
_missing = []
for _pkg in _REQUIRED:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        _missing.append(_pkg)
print(f'Kernel Python: {sys.executable}')
if _missing:
    print(f'Installing: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *_missing])
    print('RESTART KERNEL before next cell.')
else:
    print('All required packages already importable. Ready.')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import xml.etree.ElementTree as ET

PROJECT = Path.cwd().parent if Path.cwd().name == 'tutorials' else Path.cwd()
OUT = PROJECT / 'output' / 'gmpe_nga_west2'
OUT.mkdir(parents=True, exist_ok=True)

GMPE_PATH    = OUT / 'grid_gmpe_pga_sa1s.xlsx'
SHAKEMAP_XML = PROJECT / 'data' / 'grid.xml'
GMPE_NAMES   = ['ASK14', 'BSSA14', 'CB14', 'CY14']
EQ_LAT, EQ_LON = 34.213, -118.537   # Northridge epicenter

## 1. Load four-GMPE grid (Sa 1.0 s)

In [ ]:
gmpe = {}
for name in GMPE_NAMES:
    gmpe[name] = pd.read_excel(GMPE_PATH, sheet_name=name)
for name, df in gmpe.items():
    print(f'  {name:8s}  n={len(df):,}  Sa(1.0s) range = {df.Sa1s_g.min():.3f}–{df.Sa1s_g.max():.3f} g')

# All four sheets share the same grid points
grid_lats = gmpe['ASK14'].latitude.values
grid_lons = gmpe['ASK14'].longitude.values

## 2. Load USGS ShakeMap and interpolate (nearest neighbor) onto the GMPE grid

In [ ]:
def parse_shakemap(xml_path):
    """Return DataFrame with columns lat, lon, sa10 (g)."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = {'sm': root.tag.split('}')[0].strip('{')} if '}' in root.tag else {}
    fields = root.findall('.//sm:grid_field', ns) if ns else root.findall('.//grid_field')
    data_el = root.find('.//sm:grid_data', ns) if ns else root.find('.//grid_data')
    col_names = [f.get('name') for f in sorted(fields, key=lambda x: int(x.get('index')))]
    raw = data_el.text.strip().split()
    n_cols = len(col_names)
    arr = np.array(raw, dtype=float).reshape(-1, n_cols)
    df = pd.DataFrame(arr, columns=col_names)
    # ShakeMap PSA10 is in %g; divide by 100 to get g
    sa10 = df['PSA10'].values / 100.0 if 'PSA10' in df.columns else df.iloc[:, 4].values / 100.0
    return pd.DataFrame({
        'lat':  df.get('LAT', df.iloc[:, 1]).values,
        'lon':  df.get('LON', df.iloc[:, 0]).values,
        'sa10': sa10,
    })

sm = parse_shakemap(SHAKEMAP_XML)
print(f'ShakeMap: {len(sm):,} grid points, Sa(1.0s) range = {sm.sa10.min():.3f}–{sm.sa10.max():.3f} g')

In [ ]:
# Build KDTree on ShakeMap grid; query nearest-neighbor at each GMPE grid point
sm_tree = cKDTree(np.column_stack([sm.lat.values, sm.lon.values]))
_, nn_idx = sm_tree.query(np.column_stack([grid_lats, grid_lons]), k=1)
shakemap_at_grid = sm.sa10.values[nn_idx]

print(f'Interpolated ShakeMap at {len(shakemap_at_grid):,} GMPE grid points')
print(f'  Range: {shakemap_at_grid.min():.3f}–{shakemap_at_grid.max():.3f} g')

## 3. Five-panel spatial comparison

Same colour scale across all five panels so the eye can compare directly.  Star marks the epicenter.

In [ ]:
vmin = min(min(gmpe[n].Sa1s_g.min() for n in GMPE_NAMES), shakemap_at_grid.min())
vmax = max(max(gmpe[n].Sa1s_g.max() for n in GMPE_NAMES), shakemap_at_grid.max())
print(f'Shared color range: {vmin:.3f}–{vmax:.3f} g')

fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharex=True, sharey=True)
fig.suptitle('Sa(1.0 s) — 4 GMPEs vs ShakeMap (1994 Northridge)',
             fontsize=14, fontweight='bold', y=1.02)

panels = [(name, gmpe[name].Sa1s_g.values) for name in GMPE_NAMES]
panels.append(('ShakeMap (NN)', shakemap_at_grid))

for ax, (label, vals) in zip(axes, panels):
    sc = ax.scatter(grid_lons, grid_lats, c=vals, s=18,
                    cmap='plasma', vmin=vmin, vmax=vmax,
                    edgecolors='none')
    ax.scatter([EQ_LON], [EQ_LAT], marker='*', s=240, c='white',
               edgecolors='black', linewidths=1.2, zorder=10, label='epicenter')
    ax.set_title(f'{label}\nmean = {vals.mean():.3f} g', fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.grid(True, alpha=0.3, linestyle='--')

axes[0].set_ylabel('Latitude')
cbar = plt.colorbar(sc, ax=axes, fraction=0.018, pad=0.02)
cbar.set_label('Sa(1.0 s) (g)', fontsize=11)
axes[0].legend(loc='lower left', fontsize=8)

fig_path = OUT / 'comparison_5panel_Sa1s.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 4. Per-GMPE residual maps (predicted − ShakeMap)

Positive (red) = GMPE over-predicts; negative (blue) = GMPE under-predicts.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharex=True, sharey=True)
fig.suptitle('Residual: GMPE − ShakeMap nearest-neighbor (Sa 1.0 s)',
             fontsize=13, fontweight='bold', y=1.04)

residuals = {}
for ax, name in zip(axes, GMPE_NAMES):
    diff = gmpe[name].Sa1s_g.values - shakemap_at_grid
    residuals[name] = diff
    rmax = max(abs(diff.min()), abs(diff.max()))
    sc = ax.scatter(grid_lons, grid_lats, c=diff, s=18,
                    cmap='RdBu_r', vmin=-rmax, vmax=rmax,
                    edgecolors='none')
    ax.scatter([EQ_LON], [EQ_LAT], marker='*', s=200, c='black', zorder=10)
    ax.set_title(f'{name}\nbias = {diff.mean():+.3f} g, RMSE = {np.sqrt((diff**2).mean()):.3f} g',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)

axes[0].set_ylabel('Latitude')
fig_path = OUT / 'comparison_residuals.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 5. Summary statistics

Bias (mean residual), MAE, RMSE, R² between each GMPE and the ShakeMap nearest-neighbor reference, in linear-Sa space.

In [ ]:
rows = []
for name in GMPE_NAMES:
    pred = gmpe[name].Sa1s_g.values
    obs  = shakemap_at_grid
    diff = pred - obs
    bias = diff.mean()
    mae  = np.abs(diff).mean()
    rmse = np.sqrt((diff**2).mean())
    ss_res = ((obs - pred)**2).sum()
    ss_tot = ((obs - obs.mean())**2).sum()
    r2 = 1 - ss_res / ss_tot
    pearson = np.corrcoef(pred, obs)[0, 1]
    rows.append({
        'GMPE': name,
        'mean_pred (g)': pred.mean(),
        'mean_obs (g)':  obs.mean(),
        'bias (g)': bias,
        'MAE (g)':  mae,
        'RMSE (g)': rmse,
        'R²': r2,
        'Pearson r': pearson,
    })

stats = pd.DataFrame(rows).set_index('GMPE')
stats.to_csv(OUT / 'comparison_summary_stats.csv')
print(f'Saved: {OUT / "comparison_summary_stats.csv"}')
stats.round(4)

## 6. Predicted vs observed scatter

Identity line shows perfect agreement; offset above = over-prediction, below = under-prediction.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharex=True, sharey=True)
fig.suptitle('Predicted vs ShakeMap (NN) — Sa(1.0 s)',
             fontsize=13, fontweight='bold', y=1.04)

lim = max(stats['mean_pred (g)'].max(), stats['mean_obs (g)'].max()) * 2
for ax, name in zip(axes, GMPE_NAMES):
    pred = gmpe[name].Sa1s_g.values
    obs  = shakemap_at_grid
    ax.scatter(obs, pred, s=8, alpha=0.4, edgecolors='none', color='#4C72B0')
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.6, label='1:1')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('ShakeMap NN Sa(1.0 s) (g)')
    ax.set_title(f'{name}\nR² = {stats.loc[name, "R²"]:.2f}',
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='upper left', fontsize=8)

axes[0].set_ylabel('GMPE-predicted Sa(1.0 s) (g)')
fig_path = OUT / 'comparison_pred_vs_obs.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

---

**For the report (§3.2 Discussion):** the residual maps and summary table above show how each GMPE departs from the ShakeMap nearest-neighbor reference in space. Use them to argue (a) which GMPE has the smallest bias, (b) which has the most spatially uniform residuals, and (c) where the four-model ensemble would help reduce prediction uncertainty.